In [1]:
from pathlib import Path
import scanpy as sc

# Replace with your own folder path
folder = Path(
    "/michorlab/yiyang/thesis/data/drug-perturbation/"
    "Srivatsan2020_sci-Plex/h5ad"
)
input_file = folder / "sciplex3_all_cells.h5ad" # File that contain raw counts
output_file = folder / "sciplex3_pca_umap.h5ad" # File we will output

if output_file.exists():
    raise FileExistsError(output_file)

adata = sc.read_h5ad(input_file)
adata.var_names_make_unique()

adata

AnnData object with n_obs × n_vars = 799317 × 110983
    obs: 'cell', 'sample', 'Size_Factor', 'n.umi', 'hash_umis_W', 'pval_W', 'qval_W', 'top_to_second_best_ratio_W', 'top_oligo_W', 'hash_umis_P', 'pval_P', 'qval_P', 'top_to_second_best_ratio_P', 'top_oligo_P', 'rt_well', 'lig_well', 'pcr_well', 'pcr_plate', 'culture_plate', 'rt_plate', 'lig_plate', 'Combo', 'well_oligo', 'plate_oligo', 'cell_type', 'replicate', 'time_point', 'drug_dose', 'catalog_number', 'vehicle', 'dose_pattern', 'dose_character', 'dose', 'treatment', 'pathway_level_1', 'pathway_level_2', 'product_name', 'target', 'pathway'
    var: 'gene_id', 'gene_symbol'
    uns: 'dataset', 'expression', 'source_files'

In [2]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata

AnnData object with n_obs × n_vars = 798538 × 65090
    obs: 'cell', 'sample', 'Size_Factor', 'n.umi', 'hash_umis_W', 'pval_W', 'qval_W', 'top_to_second_best_ratio_W', 'top_oligo_W', 'hash_umis_P', 'pval_P', 'qval_P', 'top_to_second_best_ratio_P', 'top_oligo_P', 'rt_well', 'lig_well', 'pcr_well', 'pcr_plate', 'culture_plate', 'rt_plate', 'lig_plate', 'Combo', 'well_oligo', 'plate_oligo', 'cell_type', 'replicate', 'time_point', 'drug_dose', 'catalog_number', 'vehicle', 'dose_pattern', 'dose_character', 'dose', 'treatment', 'pathway_level_1', 'pathway_level_2', 'product_name', 'target', 'pathway', 'n_genes'
    var: 'gene_id', 'gene_symbol', 'n_cells'
    uns: 'dataset', 'expression', 'source_files'

In [3]:
# Select highly variable genes using raw counts.
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat_v3",
    n_top_genes=3000,
)

adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
work = adata[:, adata.var["highly_variable"]].copy()
del work.layers["counts"]

sc.pp.pca(work, n_comps=50, svd_solver="arpack", random_state=0)
sc.pp.neighbors(work, n_neighbors=15, n_pcs=50, random_state=0)
sc.tl.umap(work, random_state=0)

adata.obsm["X_pca"] = work.obsm["X_pca"].copy()
adata.obsm["X_umap"] = work.obsm["X_umap"].copy()
adata.obsp["distances"] = work.obsp["distances"].copy()
adata.obsp["connectivities"] = work.obsp["connectivities"].copy()
adata.uns["neighbors"] = work.uns["neighbors"].copy()
adata.uns["umap"] = work.uns["umap"].copy()

del work

In [ ]:
sc.pl.umap(adata, color="cell_type" or None, wspace=0.5)

adata.write_h5ad(output_file, compression="gzip")